# Explore the platform

The lake holds daily US equity data as Parquet: OHLCV bars, the universe of each
session, corporate actions and the FINRA short datasets. DuckDB queries the lake.
dbt builds the derived tables. Every read goes through `sdp.dal`, which returns a
lazy `DuckDBPyRelation`. Materialise a relation with `.pl()` or `.fetchall()`.
Run the cells in order, because a later section uses names from earlier cells.

In [ ]:
import polars as pl

from sdp import dal

pl.Config.set_tbl_rows(15)
pl.Config.set_fmt_str_lengths(60)

con = dal.con()   # The one connection that owns every relation.

---
## 1. What is published

A partition exists only after its audit passed. The status gives a partition count
and a date range for each event stream, and a row count for each corporate action
table.

In [ ]:
print(dal.status())

In [ ]:
# gaps() lists the XNYS sessions inside a range that have no partition.
# An empty list means the range is complete.
first, last = dal.coverage(dal.DAY_AGGS)
print("day aggregates", first, "to", last)
print("interior gaps:", dal.gaps(dal.DAY_AGGS, first, last) or "none")

---
## 2. Point-in-time

Event streams store one immutable partition per session. The corporate action
tables have no partition, because the endpoint has no `as_of` parameter. Each table
holds the present belief of the vendor. Read it with `dal.current()`, `dal.splits()`
or `dal.dividends()`.

In [ ]:
splits = dal.splits()
print("split rows:", con.sql("select count(*) from splits").fetchone()[0])
print("built from vendor pull of:",
      con.sql("select distinct vendor_pull_date from splits").fetchone()[0])

dal.splits().limit(3).pl()

In [ ]:
# Each kind of dataset refuses the accessor of the other kind.
for call, label in [
    (lambda: dal.series(dal.SPLITS), "series() on a current-state dataset"),
    (lambda: dal.current(dal.DAY_AGGS), "current() on an event stream"),
]:
    try:
        call()
    except ValueError as exc:
        print(f"{label}:\n  {exc}\n")

### What the current table cannot show

A restated factor replaces the old value and leaves no trace in `raw/`. `vendor/`
keeps every pull with its date. `ca.rebuild(dataset, pull_date)` builds the table
again as one of these pulls stated it.

In [ ]:
from sdp.ingest import massive_corporate_actions as ca

for pull, path in ca.vendor_pulls("massive_splits"):
    print(f"{pull}  {path.stat().st_size / 1e6:8.1f} MB  {path.name}")

In [ ]:
# Compare the published table with the oldest kept pull. rebuild() replaces the
# table in raw/, so this cell reads the vendor file directly.
old_pull, old_path = ca.vendor_pulls("massive_splits")[0]
con.sql(f'''
    with older as (
        select * from read_json('{old_path}',
                                format='newline_delimited', sample_size=-1)
    )
    select
        (select count(*) from splits) as rows_now,
        (select count(*) from older)  as rows_in_the_pull_of_{old_pull:%Y_%m_%d}
''').pl()

---
## 3. Restatement

A restatement is a change in what the vendor states about the past. The factor is
cumulative, so a revision changes every adjusted price before that event.
`sdp.restatement` compares two full pulls from `vendor/`.

In [ ]:
# Register the oldest and newest kept pulls as views over vendor/.
pulls = dict(ca.vendor_pulls("massive_splits"))
old_pull, new_pull = min(pulls), max(pulls)
print(f"comparing the pull of {old_pull} against {new_pull}")

for name, pull in [("older", old_pull), ("newer", new_pull)]:
    con.execute(f"""
        create or replace temp view {name} as select * from
        read_json('{pulls[pull]}', format='newline_delimited', sample_size=-1)
    """)

con.sql("""
    select (select count(*) from older) as rows_older,
           (select count(*) from newer) as rows_newer
""").pl()

The table above gives the row count of each pull. The next cell compares the two
pulls on the vendor `id`.

In [ ]:
con.sql('''
    select
        (select count(*) from older a
          where not exists (select 1 from newer b where b.id = a.id)
        ) as ids_only_in_the_old_pull,
        (select count(*) from newer b
          where not exists (select 1 from older a where a.id = b.id)
        ) as ids_only_in_the_new_pull
''').pl()

The next cell takes each `id` that is absent from the newer pull. It looks for the
same event, on `(ticker, execution_date)`, under a new `id`.

In [ ]:
con.sql('''
    with dropped as (
        select a.* from older a
        where not exists (select 1 from newer b where b.id = a.id)
    )
    select
        count(*) as ids_that_vanished,
        count(*) filter (
            exists (select 1 from newer n
                    where n.ticker = d.ticker and n.execution_date = d.execution_date)
        ) as same_event_still_present_under_a_new_id
    from dropped d
''').pl()

The vendor `id` is not stable across pulls, so compare the pulls on the event key.
The next cell counts the restated factors and the new events.

In [ ]:
# Restrict to unambiguous keys in both pulls, or the join fans out.
con.sql('''
    with ua as (select ticker, execution_date from older
                group by 1, 2 having count(*) = 1),
         ub as (select ticker, execution_date from newer
                group by 1, 2 having count(*) = 1),
         k  as (select * from ua intersect select * from ub)
    select
        (select count(*)
           from older a join newer b using (ticker, execution_date)
                        join k using (ticker, execution_date)
          where a.historical_adjustment_factor
                is distinct from b.historical_adjustment_factor) as factor_restated,
        (select count(*) from newer b
          where not exists (select 1 from older a
                            where a.ticker = b.ticker
                              and a.execution_date = b.execution_date)) as genuinely_new_events
''').pl()

A restated factor changes every adjusted price before that event. A study
records the `vendor_pull_date` of the table it used.

In [ ]:
# The restated events, newest first.
con.sql('''
    with ua as (select ticker, execution_date from older
                group by 1, 2 having count(*) = 1),
         ub as (select ticker, execution_date from newer
                group by 1, 2 having count(*) = 1),
         k  as (select * from ua intersect select * from ub)
    select a.ticker, a.execution_date, a.adjustment_type,
           a.historical_adjustment_factor as factor_before,
           b.historical_adjustment_factor as factor_after
    from older a join newer b using (ticker, execution_date)
                 join k using (ticker, execution_date)
    where a.historical_adjustment_factor
          is distinct from b.historical_adjustment_factor
    order by a.execution_date desc
    limit 15
''').pl()

---
## 4. Day aggregates

The bars are unadjusted OHLCV, with one row for each ticker and session. An adjusted
price changes when the vendor restates the past, so a partition of adjusted prices
could not stay immutable.

In [ ]:
bars = dal.day_aggs()
con.sql("select count(*) as rows, count(distinct ticker) as tickers, "
        "min(date) as first_session, max(date) as last_session from bars").pl()

In [ ]:
con.sql('''
    select date, count(*) as tickers,
           round(sum(volume * close) / 1e9, 1) as dollar_volume_bn
    from bars group by date order by date
''').pl()

`volume` is floating point, not integer. The consolidated tape carries
fractional share quantities.

In [ ]:
con.sql('''
    select count(*) as rows,
           count(*) filter (volume <> floor(volume)) as fractional_volume,
           round(100.0 * count(*) filter (volume <> floor(volume)) / count(*), 1) as pct
    from bars
''').pl()

---
## 5. Universe and identifiers

Two filters build the universe. The models apply both filters on each date
separately.

In [ ]:
names = dal.tickers_on(dal.partitions(dal.TICKERS)[-1])
con.sql('''
    select type, count(*) as n
    from names group by type order by n desc limit 12
''').pl()

In [ ]:
con.sql('''
    select
        count(*) as all_instruments,
        count(*) filter (type = 'CS') as common_stock,
        count(*) filter (type = 'CS'
                         and primary_exchange in ('XNYS','XNAS','XASE')) as after_exchange_filter
    from names
''').pl()

### The identifier problem

A ticker symbol can change, and a delisted symbol can go to a new company.
`composite_figi` is the intended key, but some rows do not have it. The next cells
measure the gaps.

In [ ]:
con.sql('''
    select
        count(*) as cs_rows,
        count(*) filter (composite_figi is null) as null_composite_figi,
        count(*) filter (share_class_figi is null) as null_share_class_figi,
        count(*) filter (cik is null) as null_cik
    from names where type = 'CS'
''').pl()

In [ ]:
# The null count of each identifier, for each instrument type.
con.sql('''
    select type, count(*) as n,
           count(*) filter (composite_figi is null) as null_figi,
           count(*) filter (cik is null) as null_cik
    from names
    where type in ('CS', 'ETF', 'ADRC', 'WARRANT', 'PFD')
    group by type order by n desc
''').pl()

The FIGI gap after the liquidity filter decides whether the fallback key matters.
Section 9 measures it.

---
## 6. Adjustment for corporate actions

The vendor `historical_adjustment_factor` is cumulative. To adjust a price on D,
find the first event after D and multiply by its factor. AAPL is the check: 2-for-1
in 2005, 7-for-1 in 2014 and 4-for-1 in 2020.

In [ ]:
splits = dal.splits()
con.sql('''
    select execution_date, adjustment_type,
           split_from, split_to,
           split_to / split_from as ratio,
           historical_adjustment_factor
    from splits where ticker = 'AAPL' order by execution_date
''').pl()

In [ ]:
# The 2005 factor must equal 1/2 * 1/7 * 1/4 = 1/56, because it includes the two
# later splits. A match confirms that the factor is cumulative.
expected = 1 / (2 * 7 * 4)
print(f"1 / (2 * 7 * 4) = {expected:.6f}")

actual = con.sql(
    "select historical_adjustment_factor from splits "
    "where ticker = 'AAPL' and execution_date = date '2005-02-28'"
).fetchone()[0]
print(f"vendor factor for 2005-02-28 = {actual}")
print("match:", round(expected, 6) == round(actual, 6))

### The boundary is strict

On the execution date, all trading is already adjusted. The join must use
`> D`, not `>= D`. An off-by-one makes one large false return per split.

In [ ]:
# The as-of join, written out.
con.sql('''
    with universe as (
        select ticker, date, close from bars where ticker = 'AAPL'
    )
    select u.date, u.close,
           (select s.historical_adjustment_factor
              from splits s
             where s.ticker = u.ticker
               and s.execution_date > u.date
             order by s.execution_date
             limit 1) as split_factor
    from universe u
    order by u.date
    limit 10
''').pl()

A null factor means that no split follows that date. The price needs no split
adjustment.

---
## 7. Audit invariants on live data

Confirm that the properties the audits enforce at ingest hold on what is published.

In [ ]:
# Split classification: forward > 1, reverse < 1, stock dividend > 1.
con.sql('''
    select adjustment_type,
           count(*) as n,
           min(split_to / split_from) as min_ratio,
           max(split_to / split_from) as max_ratio
    from splits group by adjustment_type order by n desc
''').pl()

In [ ]:
# The share of each adjustment type among all splits.
con.sql('''
    select adjustment_type, count(*) as n,
           round(100.0 * count(*) / sum(count(*)) over (), 1) as pct
    from splits group by adjustment_type order by n desc
''').pl()

In [ ]:
# A zero factor can be correct. The RYCEF C shares are not fungible, so no valid
# adjustment exists.
con.sql('''
    select ticker, execution_date, adjustment_type, split_from, split_to,
           historical_adjustment_factor
    from splits
    where historical_adjustment_factor <= 0
    order by execution_date desc
    limit 10
''').pl()

In [ ]:
# A null dividend factor means that the vendor had no price on the ex-date.
divs = dal.dividends()
con.sql('''
    select count(*) as rows,
           count(*) filter (historical_adjustment_factor is null) as null_factor,
           round(100.0 * count(*) filter (historical_adjustment_factor is null)
                 / count(*), 1) as pct_null,
           count(*) filter (currency is not null and currency <> 'USD') as not_usd
    from divs
''').pl()

In [ ]:
# Compare the null rate of the tickers on the ingested tape with the rest.
con.sql('''
    with traded as (select distinct ticker from bars)
    select
        case when d.ticker in (select ticker from traded)
             then 'in day aggs' else 'not in day aggs' end as group_,
        count(*) as rows,
        round(100.0 * count(*) filter (d.historical_adjustment_factor is null)
              / count(*), 1) as pct_null_factor
    from divs d group by 1
''').pl()

The table above gives the null rate on the tape and off the tape. The rows off the
tape are mostly foreign issuers, OTC names and fund classes. `adj_close_total` is
the default column, because only the null rate on the tape affects a price.

---
## 8. The universe over the full history

The first cell prints the session count and the date range of `stg_universe`. Then
it shows the size of the universe on each date.

In [ ]:
wh = dal.warehouse()   # A read-only connection to the dbt warehouse.

n_sessions, first, last = wh.sql(
    "select count(distinct date), min(date), max(date) from main_staging.stg_universe"
).fetchone()
print(f"{n_sessions:,} sessions, {first} to {last}")

wh.sql('''
    select date, count(*) filter (in_universe) as names
    from main_staging.stg_universe
    group by 1 order by 1
''').pl()

In [ ]:
# The size of the universe for each year. The target is 1,000 to 2,000 names.
median_names, first_session = wh.sql('''
    with per_date as (
        select date, count(*) filter (in_universe) as n
        from main_staging.stg_universe group by 1
    )
    select median(n) filter (n > 0), min(date) filter (n > 0) from per_date
''').fetchone()
print(f"median size {median_names:,.0f} names, first non-empty session {first_session}")

wh.sql('''
    with per_date as (
        select date, count(*) filter (in_universe) as n
        from main_staging.stg_universe group by 1
    )
    select date_trunc('year', date) as year,
           round(avg(n)) as avg_names,
           min(n) as min_names,
           max(n) as max_names
    from per_date group by 1 order by 1
''').pl()

Compare the median with the target of 1,000 to 2,000 names. `min_dollar_volume` is
the dial that sets the size. A name needs `min_days_since_first_bar` sessions of
history, so the first sessions of the lake have an empty universe. The usable
sample starts at the first non-empty session that the cell above prints.

In [ ]:
# The names that remain after each filter on the last session. Each filter is a
# column.
wh.sql('''
    select
        count(*)                                  as rows,
        count(*) filter (passes_instrument)       as after_instrument,
        count(*) filter (passes_instrument and passes_price)      as and_price,
        count(*) filter (passes_instrument and passes_price
                         and passes_adv)          as and_adv,
        count(*) filter (in_universe)             as in_universe
    from main_staging.stg_universe
    where date = (select max(date) from main_staging.stg_universe)
''').pl()

---
## 9. Open questions

`python -m sdp.diagnostics` runs these against the whole lake.

### The identifier

A ticker can go to a different company. The size of this effect decides whether a
simple key works.

In [ ]:
names = dal.tickers()

con.sql('''
    with per_ticker as (
        select ticker, count(distinct composite_figi) as figis
        from names where type = 'CS' and composite_figi is not null
        group by 1
    )
    select count(*) as cs_tickers,
           count(*) filter (figis > 1) as more_than_one_figi,
           count(*) filter (figis > 2) as more_than_two
    from per_ticker
''').pl()

The table above counts the CS tickers that carry more than one FIGI across the
window. The next cell measures the fallback rate after the liquidity screen, which
decides the key rule.

In [ ]:
# A filled FIGI counts as on the FIGI. The coalesce counts a null key_rule as off the
# FIGI, so no row drops silently.
wh.sql('''
    with k as (
        select in_universe, not coalesce(key_rule like 'share_class_figi%', false) as off
        from main_staging.stg_universe
        where type = 'CS'
    )
    select
        count(*) as cs_rows,
        round(100.0 * count(*) filter (off) / count(*), 2) as pct_not_on_figi,
        count(*) filter (in_universe) as after_the_screen,
        round(100.0 * count(*) filter (in_universe and off)
              / nullif(count(*) filter (in_universe), 0), 2) as pct_not_on_figi_in_universe
    from k
''').pl()

The table above gives the share of rows that are not keyed on `share_class_figi`,
before and after the screen. `stg_tickers` builds `security_key` as a coalesce and
records the rule in `key_rule`. A FIGI that the vendor drops for some dates is filled
from the same ticker and CIK (`share_class_figi_filled`). Report each result with and
without the fallback rows.

### The ambiguous event key

Neither `(ticker, execution_date)` nor `(ticker, ex_dividend_date)` is unique.

In [ ]:
splits = dal.splits()
divs = dal.dividends()

# A null factor counts as one more value, so a null beside a number is a
# disagreement.
con.sql('''
    with per_key as (
        select 'split' as kind, ticker, execution_date as event_date,
               count(*) as rows,
               count(distinct historical_adjustment_factor)
                 + (count(*) > count(historical_adjustment_factor))::int as factors
        from splits group by 1, 2, 3
        union all
        select 'dividend', ticker, ex_dividend_date,
               count(*),
               count(distinct historical_adjustment_factor)
                 + (count(*) > count(historical_adjustment_factor))::int
        from divs group by 1, 2, 3
    )
    select kind,
           count(*) as distinct_keys,
           count(*) filter (rows > 1) as duplicated,
           count(*) filter (rows > 1 and factors > 1) as and_disagreeing
    from per_key group by kind order by kind desc
''').pl()

The table above counts the duplicated keys of each kind, and the duplicated keys
whose rows give different factors. An as-of join against the raw rows fans out on
a duplicated key, so `stg_corporate_actions` collapses each key to one row first.

In [ ]:
# The trap, demonstrated. Joining prices to the raw splits on the event key
# multiplies rows wherever the key is duplicated.
con.sql('''
    with one_name as (
        select ticker, execution_date
        from splits group by 1, 2 having count(*) > 1 limit 1
    )
    select s.ticker, s.execution_date, s.split_from, s.split_to,
           s.historical_adjustment_factor
    from splits s join one_name using (ticker, execution_date)
''').pl()

---
## 10. Restatement detail

`python -m sdp.restatement` compares the oldest and newest vendor pulls.

In [ ]:
from sdp import restatement

# sdp.restatement reads vendor/, so it keeps working however raw/ is stored.
print(restatement.diff(dal.SPLITS))

The `id diff` line counts on the vendor `id`, and the `event diff` line counts on
the event key. The `id churn only` line gives the events that are present in both
pulls under a new `id`.

In [ ]:
# The event diff, written out, to show what the module does internally.
con.sql('''
    with ka as (select distinct ticker, execution_date from older),
         kb as (select distinct ticker, execution_date from newer)
    select
        (select count(*) from ka where not exists
            (select 1 from kb where kb.ticker = ka.ticker
                                and kb.execution_date = ka.execution_date)) as events_gone,
        (select count(*) from kb where not exists
            (select 1 from ka where ka.ticker = kb.ticker
                                and ka.execution_date = kb.execution_date)) as events_new
''').pl()

Compare these counts on the event key with the counts on the `id` in section 3. The
difference is `id` churn, not a change in the events.

### Mechanical and real restatement

A new dividend changes every earlier factor by design. The next cell separates this
mechanical change from a correction by the vendor.

In [ ]:
d_pulls = dict(ca.vendor_pulls("massive_dividends"))
d_old, d_new = min(d_pulls), max(d_pulls)
for name, pull in [("d_older", d_old), ("d_newer", d_new)]:
    con.execute(f"""
        create or replace temp view {name} as select * from
        read_json('{d_pulls[pull]}', format='newline_delimited', sample_size=-1)
    """)

con.sql(f'''
    with ua as (select ticker, ex_dividend_date from d_older
                group by 1, 2 having count(*) = 1),
         ub as (select ticker, ex_dividend_date from d_newer
                group by 1, 2 having count(*) = 1),
         k  as (select * from ua intersect select * from ub),
         changed as (
             select a.ticker
             from d_older a join d_newer b using (ticker, ex_dividend_date)
                            join k using (ticker, ex_dividend_date)
             where a.historical_adjustment_factor
                   is distinct from b.historical_adjustment_factor
         ),
         went_ex as (
             select distinct ticker from d_newer
             where ex_dividend_date > date '{d_old}'
               and ex_dividend_date <= date '{d_new}'
         )
    select count(*) as restated,
           count(*) filter (ticker in (select ticker from went_ex)) as mechanical,
           count(*) filter (ticker not in (select ticker from went_ex)) as vendor_revision,
           round(100.0 * count(*) filter (ticker in (select ticker from went_ex))
                 / nullif(count(*), 0), 1) as pct_mechanical
    from changed
''').pl()

`mechanical` counts the restated rows of tickers that went ex between the two pulls.
`vendor_revision` counts the other restated rows, where the vendor changed a factor
with no new event.

## 11. Signal IC

The table gives the daily Spearman IC of each signal over development, from
`main_marts.mart_signal_ic_summary`. The target `return` is the forward return over the
full price series of each security. The target `residual` is the part of that return
that the style and industry model does not explain, so a residual IC is a prediction
beyond the known factors. `t_nw` is the Newey-West t, which allows for the overlap of
the forward windows. `t_naive` assumes independent days and is too high for a long
horizon. `folds_same_sign` counts the development folds that agree in sign with the
whole. The holdout is not in this table.

In [ ]:
ic = wh.sql('''
    select signal, target, horizon, n_days,
           round(mean_ic, 4) as mean_ic,
           round(t_nw, 2)    as t_nw,
           round(t_naive, 2) as t_naive,
           folds_same_sign
    from main_marts.mart_signal_ic_summary
    where period = 'development'
    order by signal, target, horizon
''').pl()

with pl.Config(tbl_rows=len(ic)):
    print(ic)

To test the signals against a higher liquidity floor, build the models again with
a larger `min_dollar_volume`:

```bash
python -m sdp.transform build --vars '{min_dollar_volume: 10000000}'
```

Then run `wh.close()` and `wh = dal.warehouse()`, and run this section again. DuckDB
keeps the old build while a connection to it is open, so close `wh` first. If the
reversal IC falls when the floor rises, the signal is an illiquidity premium that a
trade cannot capture.